# 🧪 Experiments: 5 Models × 3 Datasets

This notebook runs all 15 model-dataset combinations (`GATv2`, `GraphSAGE`, `H2GCN`, `MLP`, `Random` × `WikiCS`, `Amazon Computers`, `Roman-empire`) with:
- Validation-only hyperparameter tuning (grid search)
- Multi-seed evaluation (5 seeds for mean ± std)
- Result tables, training curves, and saved checkpoints

> **Tip:** For unattended GPU execution on Colab or NVIDIA GPU, you can also run `python run_project.py --device cuda` from the project root.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.datasets import load_dataset, get_all_dataset_names
from src.models import create_model, get_all_model_names
from src.train import train, train_multi_seed, tune_hyperparameters, get_quick_param_grid
from src.evaluate import compute_all_metrics
from src.utils import set_seed, get_device, count_parameters

device = get_device()
set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 150

os.makedirs('../results/tables', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)
os.makedirs('../results/models', exist_ok=True)

MODEL_DISPLAY = {'gatv2': 'GATv2', 'graphsage': 'GraphSAGE', 'h2gcn': 'H2GCN', 'mlp': 'MLP', 'random': 'Random'}
MODEL_COLORS = {'gatv2': '#1b4965', 'graphsage': '#2a9d8f', 'h2gcn': '#e76f51', 'mlp': '#ca6702', 'random': '#6c757d'}
TRAINABLE_MODELS = ['gatv2', 'graphsage', 'h2gcn', 'mlp']

print('Setup complete!')

## 1. Load All Datasets

In [ ]:
datasets = {}
dataset_names = ['amazon-computers', 'wikics', 'roman-empire']
for name in dataset_names:
    datasets[name] = load_dataset(name, root='../data')
    print(f"{name}: {datasets[name].num_nodes} nodes, {datasets[name].num_node_features} features, {datasets[name].num_classes} classes")

model_names = get_all_model_names()
print(f"\nModels: {model_names}")
print(f"Datasets: {dataset_names}")
print(f"Total pairs: {len(model_names)} x {len(dataset_names)} = {len(model_names) * len(dataset_names)}")

## 2. Model Parameter Counts

In [ ]:
param_table = []
for dname in dataset_names:
    data = datasets[dname]
    for mname in model_names:
        model = create_model(mname, data.num_node_features, data.num_classes)
        total, trainable = count_parameters(model)
        param_table.append({
            'Model': MODEL_DISPLAY[mname],
            'Dataset': data.dataset_name,
            'Total Params': f"{total:,}",
            'Trainable Params': f"{trainable:,}",
        })

pd.DataFrame(param_table)

## 3. Hyperparameter Tuning (Validation Accuracy Only)

In [ ]:
use_full_grid = torch.cuda.is_available()  # Auto-use full grid on GPU
best_params = {}

for dname in dataset_names:
    data = datasets[dname]
    best_params[dname] = {}
    for mname in model_names:
        if mname == 'random':
            best_params[dname][mname] = {}
            continue
        param_grid = None if use_full_grid else get_quick_param_grid(mname)
        params, val_acc, _ = tune_hyperparameters(
            mname, data, device,
            param_grid=param_grid,
            max_epochs=150 if use_full_grid else 60,
            patience=20 if use_full_grid else 10,
            verbose=True,
        )
        best_params[dname][mname] = params

with open('../results/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print('Saved results/best_params.json')

## 4. Multi-Seed Evaluation (5 Seeds)

In [ ]:
all_results = {}
seeds = [42, 123, 456, 789, 1024]

for dname in dataset_names:
    data = datasets[dname]
    all_results[dname] = {}
    for mname in model_names:
        params = best_params[dname][mname].copy()
        lr = params.pop('lr', 0.005)
        wd = params.pop('weight_decay', 5e-4)
        max_ep = 1 if mname == 'random' else 500
        results = train_multi_seed(
            mname, data, device,
            seeds=seeds,
            lr=lr, weight_decay=wd,
            max_epochs=max_ep, patience=50,
            verbose=True,
            **params,
        )
        all_results[dname][mname] = results

## 5. Results Table

In [ ]:
rows = []
for mname in model_names:
    row = {'Model': MODEL_DISPLAY[mname]}
    for dname in dataset_names:
        r = all_results[dname][mname]
        ds_label = datasets[dname].dataset_name
        row[f'{ds_label} Acc'] = f"{r['mean_accuracy']:.4f} ± {r['std_accuracy']:.4f}"
        row[f'{ds_label} F1'] = f"{r['mean_f1']:.4f} ± {r['std_f1']:.4f}"
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv('../results/tables/main_results.csv', index=False)
results_df

## 6. Training Curves (Seed 42)

In [ ]:
fig, axes = plt.subplots(len(TRAINABLE_MODELS), len(dataset_names), figsize=(16, 12), constrained_layout=True)

for j, dname in enumerate(dataset_names):
    for i, mname in enumerate(TRAINABLE_MODELS):
        ax = axes[i][j]
        h = all_results[dname][mname]['per_seed_histories'][0]
        epochs = range(1, len(h['train_acc']) + 1)
        ax.plot(epochs, h['train_acc'], color=MODEL_COLORS[mname], alpha=0.55, linestyle='--', label='Train Acc')
        ax.plot(epochs, h['val_acc'], color=MODEL_COLORS[mname], alpha=0.95, linewidth=2, label='Val Acc')
        if h['best_epoch'] > 0:
            ax.axvline(h['best_epoch'], color='gray', linestyle=':', alpha=0.6)
        ax.set_title(f"{MODEL_DISPLAY[mname]} on {datasets[dname].dataset_name}", fontsize=10, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend(fontsize=8, loc='lower right')
        ax.set_ylim(0, 1.02)

fig.suptitle('Training & Validation Curves (Seed 42)', fontsize=14, fontweight='bold')
fig.savefig('../results/plots/training_curves.png', dpi=180)
plt.show()

## 7. Performance Comparison Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), constrained_layout=True)
x = np.arange(len(dataset_names))
width = 0.16

for ax, key_mean, key_std, ylabel, title in [
    (axes[0], 'mean_accuracy', 'std_accuracy', 'Test Accuracy', 'Test Accuracy Comparison'),
    (axes[1], 'mean_f1', 'std_f1', 'Macro F1-Score', 'Macro F1-Score Comparison'),
]:
    for i, mname in enumerate(model_names):
        means = [all_results[d][mname][key_mean] for d in dataset_names]
        stds = [all_results[d][mname][key_std] for d in dataset_names]
        ax.bar(x + i * width, means, width, yerr=stds, label=MODEL_DISPLAY[mname], color=MODEL_COLORS[mname], capsize=3, edgecolor='white')
    ax.set_xticks(x + width * (len(model_names) - 1) / 2)
    ax.set_xticklabels([datasets[d].dataset_name for d in dataset_names])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)

fig.savefig('../results/plots/benchmark_comparison.png', dpi=180)
plt.show()

## 8. Save Seed-42 Checkpoints (for Milestone 3 Notebooks)

In [ ]:
for dname in dataset_names:
    data = datasets[dname]
    ds_folder = data.dataset_name.lower().replace(' ', '_')
    for mname in TRAINABLE_MODELS:
        set_seed(42)
        params = best_params[dname][mname].copy()
        lr = params.pop('lr', 0.005)
        wd = params.pop('weight_decay', 5e-4)
        model = create_model(mname, data.num_node_features, data.num_classes, **params).to(device)
        data_dev = data.to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        history = train(model, data_dev, optimizer, max_epochs=500, patience=50, verbose=False)

        save_path = f'../results/models/{ds_folder}/{mname}/seed_42.pt'
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        torch.save({
            'model_state': model.state_dict(),
            'model_name': mname,
            'dataset': data.dataset_name,
            'seed': 42,
            'model_kwargs': params,
            'lr': lr,
            'weight_decay': wd,
            'best_val_accuracy': history['best_val_acc'],
        }, save_path)
        print(f"Saved {save_path} (val_acc: {history['best_val_acc']:.4f})")